# `judge.py`

This notebook is organized into logical, executable sections from the corresponding Python source file.

## Setup

Load dependencies, environment values, and the verification prompt.

In [ ]:
﻿"""Minimal Groq verifier for evidence-based answers."""

import json
import os
from pathlib import Path
import requests
from dotenv import load_dotenv

load_dotenv(Path(__file__).parent / ".env")
PROMPT = (Path(__file__).parent / "prompt.txt").read_text(encoding="utf-8")




## Evidence-based judge

Submit the question, answer, and evidence to Groq and handle failures safely.

In [ ]:
def verify_answer(question: str, answer: str, evidence: list[dict]) -> dict:
    if not os.getenv("GROQ_API_KEY"):
        return {"overall_verdict": "unverified", "reason": "GROQ_API_KEY is missing"}

    block = "\n".join(
        f"[{index}] doc_id={item['doc_id']}\n{item['text']}"
        for index, item in enumerate(evidence, 1)
    )
    message = PROMPT.format(question=question, answer=answer, evidence=block)
    try:
        response = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
            json={
                "model": os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
                "temperature": 0,
                "max_tokens": 2048,
                "messages": [{"role": "user", "content": message}],
            },
            timeout=60,
        )
        response.raise_for_status()
        payload = response.json()
        content = payload["choices"][0]["message"]["content"]
        return json.loads(content)
    except (requests.RequestException, ValueError, KeyError, TypeError) as error:
        return {
            "overall_verdict": "unverified",
            "reason": f"Groq verification failed: {error}",
            "evidence": evidence,
        }
